# Thesis Pipeline Orchestrator
This notebook starts the Neo4j environment, optionally rebuilds the dataset, selects an optimized historical/detection window,
and runs the full label-aware and label-agnostic FastRP analyses. Artifacts are archived under `thesis_results/` for thesis use.

In [1]:
# Global configuration for the thesis analysis pipeline
from pathlib import Path
from datetime import datetime
import glob
import json
import shutil
import pandas as pd
import numpy as np

from CART import Controller

# Window configuration (optimized from prior sweep)
HISTORICAL_WINDOW_HOURS = 48
DETECTION_WINDOW_HOURS = 24
EMBEDDING_DIM = 128

# Pipeline control flags
REBUILD_DATABASE = False    # Set True only if you need fresh data from source
RUN_WINDOW_SWEEP = False    # ⚠️ EXPENSIVE - Already optimized, keep False
USE_OPTIMIZED_WINDOW = True # Use pre-computed optimal windows (48h/24h)

# Experiment selection
RUN_LABEL_AWARE = True      # Run MITRE ATT&CK-aware experiment
RUN_LABEL_AGNOSTIC = True   # Run structural (label-agnostic) experiment
LABEL_AGNOSTIC_LIMIT = None # Optional cap for label-agnostic recon events

# Post-processing
SHUTDOWN_AFTER_RUN = False        # Stop container when notebook completes
REFRESH_CONNECTS_EXPORT = True    # Force CONNECTS export refresh before visualization
GENERATE_THESIS_ARTIFACTS = True  # Run artifact generator after pipeline completes

OUTPUT_DIR = Path('thesis_results')
OUTPUT_DIR.mkdir(exist_ok=True)

print("="*80)
print("THESIS PIPELINE CONFIGURATION")
print("="*80)
print(f"Window: {HISTORICAL_WINDOW_HOURS}h historical, {DETECTION_WINDOW_HOURS}h detection")
print(f"Rebuild Database: {REBUILD_DATABASE}")
print(f"Run Window Sweep: {RUN_WINDOW_SWEEP} (⚠️ Skip unless re-optimizing)")
print(f"Label-Aware: {RUN_LABEL_AWARE}")
print(f"Label-Agnostic: {RUN_LABEL_AGNOSTIC}")
print(f"Generate Artifacts: {GENERATE_THESIS_ARTIFACTS}")
print("="*80)

THESIS PIPELINE CONFIGURATION
Window: 48h historical, 24h detection
Rebuild Database: False
Run Window Sweep: False (⚠️ Skip unless re-optimizing)
Label-Aware: True
Label-Agnostic: True
Generate Artifacts: True


In [2]:
# Start or connect to the shared Neo4j controller container
controller = Controller()

Controller initialized.
Analysis Controller (inherits Neo4jConnection) is configured and ready.


In [3]:

status = controller.status()
if not status.get('running'):
    print('Starting Neo4j container...')
    controller.start()
else:
    print('Neo4j container already running.')

if not controller.connect():
    raise RuntimeError('Could not connect to Neo4j; check container logs.')
print('Controller connected to Neo4j.')

Docker requires elevated privileges. Switching to sudo...
Container 'neo4j_thesis_server': exists, running; driver_connected=False
Neo4j container already running.
✓ Successfully connected to the Neo4j database.
Controller connected to Neo4j.
Container 'neo4j_thesis_server': exists, running; driver_connected=False
Neo4j container already running.
✓ Successfully connected to the Neo4j database.
Controller connected to Neo4j.


In [4]:
# Optional database rebuild (downloads the dataset and reimports into Neo4j)
if REBUILD_DATABASE:
    print('Rebuilding database with full dataset...')
    controller.build_database(rebuild=True)
else:
    print('Skipping database rebuild; set REBUILD_DATABASE=True to rebuild.')

Skipping database rebuild; set REBUILD_DATABASE=True to rebuild.


In [5]:
# Optional window sweep to regenerate comparison CSVs
if RUN_WINDOW_SWEEP:
    import optimize_windows
    optimize_windows.input = lambda prompt='': None
    optimize_windows.main()
else:
    print('Skipping window sweep; set RUN_WINDOW_SWEEP=True to execute optimize_windows.py.')

Skipping window sweep; set RUN_WINDOW_SWEEP=True to execute optimize_windows.py.


In [6]:
# Evaluate available window optimization outputs and optionally update window selection
records = []
for path in sorted(glob.glob('window_opt_*_method_comparison.csv')):
    parts = Path(path).stem.split('_')
    hist = int(parts[2])
    det = int(parts[3])
    df = pd.read_csv(path)
    if 'FastRP Embedding' not in df['Method'].values:
        continue
    row = df[df['Method'] == 'FastRP Embedding'].iloc[0]
    pred_path = path.replace('_method_comparison.csv', '_pivot_predictions.csv')
    try:
        pred_df = pd.read_csv(pred_path, usecols=['became_pivot'])
        pivot_rate = pred_df['became_pivot'].mean() * 100
        sample_count = len(pred_df)
        pivot_count = pred_df['became_pivot'].sum()
    except Exception:
        pivot_rate = np.nan
        sample_count = np.nan
        pivot_count = np.nan
    records.append({
        'historical_hours': hist,
        'detection_hours': det,
        'auc_roc': row['AUC-ROC'],
        'auc_pr': row['AUC-PR'],
        'f1': row['F1-Score'],
        'pivot_rate': pivot_rate,
        'samples': sample_count,
        'pivot_count': pivot_count
    })

if records:
    window_df = pd.DataFrame(records).sort_values(
        ['auc_roc', 'historical_hours', 'detection_hours'],
        ascending=[False, True, True]
    )
    print(window_df.to_string(index=False))
    if USE_OPTIMIZED_WINDOW:
        best_row = window_df.iloc[0]
        HISTORICAL_WINDOW_HOURS = int(best_row['historical_hours'])
        DETECTION_WINDOW_HOURS = int(best_row['detection_hours'])
        print(f"Using best window: hist={HISTORICAL_WINDOW_HOURS}h, det={DETECTION_WINDOW_HOURS}h")
else:
    print('No window optimization files found; retaining configured windows.')

print(f'Analysis window configuration: hist={HISTORICAL_WINDOW_HOURS}h, det={DETECTION_WINDOW_HOURS}h')

 historical_hours  detection_hours  auc_roc   auc_pr       f1  pivot_rate  samples  pivot_count
               48               24 0.869766 0.990443 0.973563   94.848738    28692        27214
               12               48 0.720415 0.981617 0.973563   94.848738    28692        27214
               24               12 0.715505 0.981390 0.973563   94.848738    28692        27214
               24               48 0.710609 0.981100 0.973563   94.848738    28692        27214
               12               12 0.673137 0.978159 0.973563   94.848738    28692        27214
               48               12 0.652763 0.976906 0.973563   94.848738    28692        27214
               48               48 0.644939 0.976127 0.973563   94.848738    28692        27214
               24               24 0.633053 0.975107 0.973563   94.848738    28692        27214
               12               24 0.591928 0.971782 0.973563   94.848738    28692        27214
Using best window: hist=48h, det=24h
Ana

In [ ]:
# Prepare SubnetPivotAnalyzer with full-corpus reconnaissance sampling
analyzer = controller.SubnetPivotAnalyzer
if not analyzer.connect():
    raise RuntimeError('Analyzer could not connect to Neo4j.')

def patch_full_recon_sampling(analyzer_obj, label_agnostic_limit=None):
    original_fn = analyzer_obj.identify_reconnaissance_victims_by_subnet

    def patched(self, use_labels: bool, historical_window_hours: int):
        print('\n--- Identifying Reconnaissance Victims by Subnet (full corpus) ---')
        with self.driver.session(database=self.database) as session:
            if use_labels:
                query = (
                    "MATCH (a:IP)-[r:CONNECTS]->(v:IP)\n"
                    "WHERE r.is_attack = 1 AND r.tactic = 'Reconnaissance'\n"
                    "WITH DISTINCT v.subnet as victim_subnet, r.timestamp as recon_time\n"
                    "ORDER BY recon_time\n"
                    "RETURN victim_subnet, recon_time"
                )
            else:
                query = (
                    "MATCH (a:IP)-[r1:CONNECTS]->(v:IP)\n"
                    "WHERE exists { (v)-[:CONNECTS]->() }\n"
                    "WITH DISTINCT v.subnet as victim_subnet, r1.timestamp as recon_time\n"
                    "ORDER BY recon_time\n"
                    "RETURN victim_subnet, recon_time"
                )
                if label_agnostic_limit is not None:
                    query += f'\nLIMIT {int(label_agnostic_limit)}'
            result = session.run(query).data()
        print(f"  ✓ Found {len(result):,} reconnaissance events")
        return result

    analyzer_obj.identify_reconnaissance_victims_by_subnet = patched.__get__(analyzer_obj, analyzer_obj.__class__)
    return original_fn

original_recon_fn = patch_full_recon_sampling(analyzer, label_agnostic_limit=LABEL_AGNOSTIC_LIMIT)

: 

In [ ]:
# Run selected experiments with the configured window
run_modes = []
if RUN_LABEL_AWARE:
    run_modes.append('label_aware')
if RUN_LABEL_AGNOSTIC:
    run_modes.append('label_agnostic')

if not run_modes:
    raise ValueError('No experiments selected; enable RUN_LABEL_AWARE and/or RUN_LABEL_AGNOSTIC.')

def mode_artifacts_available(prefix: str) -> bool:
    pivot_path = Path(f"{prefix}_pivot_predictions.csv")
    method_path = Path(f"{prefix}_method_comparison.csv")
    missing = [p.name for p in (pivot_path, method_path) if not p.exists()]
    if missing:
        print(f"  ⚠ Missing artifacts for {prefix}: {', '.join(missing)}")
        return False
    return True

executed_prefixes = []
overall_start = datetime.utcnow()
try:
    for mode in run_modes:
        print('\n' + '=' * 80)
        print(f"Executing {mode.replace('_', ' ').title()} experiment")
        print('=' * 80)
        mode_start = datetime.utcnow()
        analyzer.run_full_analysis(
            mode=mode,
            historical_window_hours=HISTORICAL_WINDOW_HOURS,
            detection_window_hours=DETECTION_WINDOW_HOURS,
            embedding_dim=EMBEDDING_DIM
        )
        mode_finish = datetime.utcnow()
        if mode_artifacts_available(mode):
            executed_prefixes.append(mode)
        else:
            print(f"  ⚠ Skipping downstream steps for {mode}; required artifacts not produced.")
        print(f"{mode} runtime: {(mode_finish - mode_start).total_seconds():.1f} seconds")

    if {'label_aware', 'label_agnostic'}.issubset(set(executed_prefixes)):
        analyzer.compare_analysis_modes()
    else:
        print("  ⚠ Comparison skipped; ensure both modes complete successfully before comparing.")
finally:
    analyzer.identify_reconnaissance_victims_by_subnet = original_recon_fn
    overall_finish = datetime.utcnow()
    print(f"Total analysis runtime: {(overall_finish - overall_start).total_seconds():.1f} seconds")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name)'



Executing Label Aware experiment

SUBNET-AWARE PIVOT PREDICTION WITH FASTRP EMBEDDINGS
  Mode: LABEL_AWARE
  Historical window: 48 hours
  Detection window: 24 hours
  Embedding dimension: 128

--- Adding Subnet Labels to IP Nodes ---
  Adding subnet strings...
  Creating subnet ID mapping (using deterministic hash)...
  ✓ Assigned deterministic numeric IDs to 21 subnets
  ✓ Subnet indices created

RUNNING LABEL-AWARE ANALYSIS

--- Creating Graph Projection: pivot_graph_labeled ---

--- Dropping All Existing Graph Projections ---
  ✓ Dropped existing projection
  ✓ Created label-aware projection: pivot_graph_labeled
    Nodes: 357
    Relationships: 3,797,226

--- Computing FastRP Embeddings (dim=128) ---
  Computing label-aware embeddings (single pass with weights)...
  ✓ Created label-aware projection: pivot_graph_labeled
    Nodes: 357
    Relationships: 3,797,226

--- Computing FastRP Embeddings (dim=128) ---
  Computing label-aware embeddings (single pass with weights)...
  ✓ Lab

  Processing Training events: 100%|██████████| 28692/28692 [00:04<00:00, 6966.24subnet/s]



  Training Results:
    Valid samples: 28,692
    Pivots: 28,004 (97.6%)

--- Processing Test Set ---
  Pre-fetching all subnet features and pivot data...
    ✓ Loaded features for 21 subnets
    Fetching LATERAL MOVEMENT attacks for 14 unique subnets...
    Time range: 1710685406.49361 to 1711669990.118523 (273.5 hours)
    ✓ Fetched 364658 lateral movement attacks
    Processing pivot behaviors in memory...
    ✓ Fetched 364658 lateral movement attacks
    Processing pivot behaviors in memory...
    ✓ Loaded pivot behaviors for 28692 events
    ✓ 27214 (94.8%) are TRUE PIVOTS (lateral movement detected)
  Processing 28692 events in memory...
    ✓ Loaded pivot behaviors for 28692 events
    ✓ 27214 (94.8%) are TRUE PIVOTS (lateral movement detected)
  Processing 28692 events in memory...


  Processing Testing events: 100%|██████████| 28692/28692 [00:03<00:00, 7376.63subnet/s]




  Test Results:
    Valid samples: 28,692
    Pivots: 27,214 (94.8%)

--- Statistical Analysis ---

  FastRP Similarity Statistics:
    Pivots:     mean=0.4651, std=0.2547
    Non-pivots: mean=0.2420, std=0.1240
    Difference: 0.2232

  Welch's t-test: t=62.4186, p=0.000000
  ✓ STATISTICALLY SIGNIFICANT (p < 0.05)
  Cohen's d: 1.1140 (large effect)
  Mann-Whitney U: U=28663752, p=0.000000

--- Baseline Comparison ---

--- Statistical Analysis ---

  FastRP Similarity Statistics:
    Pivots:     mean=0.4651, std=0.2547
    Non-pivots: mean=0.2420, std=0.1240
    Difference: 0.2232

  Welch's t-test: t=62.4186, p=0.000000
  ✓ STATISTICALLY SIGNIFICANT (p < 0.05)
  Cohen's d: 1.1140 (large effect)
  Mann-Whitney U: U=28663752, p=0.000000

--- Baseline Comparison ---

  Applying multiple testing correction (Benjamini-Hochberg)...
  ✓ Benjamini-Hochberg correction applied

METHOD COMPARISON
             Method  AUC-ROC  AUC-PR  Accuracy  Precision  Recall  F1-Score  p_value  cohens_d  p_v

In [ ]:
# Collect and archive artifacts for this run
run_stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
run_dir = OUTPUT_DIR / f'run_{run_stamp}_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}'
run_dir.mkdir(parents=True, exist_ok=True)

def add_window_tag(name: str) -> str:
    if name.startswith('label_aware_'):
        return name.replace('label_aware_', f"label_aware_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}_", 1)
    if name.startswith('label_agnostic_'):
        return name.replace('label_agnostic_', f"label_agnostic_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}_", 1)
    return name

patterns = [f"{prefix}_*" for prefix in executed_prefixes]
patterns.append('mode_comparison.png')

moved = []
for pattern in patterns:
    for src_path in Path('.').glob(pattern):
        if not src_path.is_file():
            continue
        dest_name = add_window_tag(src_path.name)
        dest_path = run_dir / dest_name
        shutil.move(str(src_path), dest_path)
        moved.append(dest_path)
        print(f'Moved {src_path.name} -> {dest_path}')

metadata = {
    'timestamp_utc': run_stamp,
    'historical_window_hours': HISTORICAL_WINDOW_HOURS,
    'detection_window_hours': DETECTION_WINDOW_HOURS,
    'embedding_dim': EMBEDDING_DIM,
    'executed_prefixes': executed_prefixes,
    'artifacts': [str(path.name) for path in moved]
}
(run_dir / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
print(f'Archived run artifacts in {run_dir}')

In [ ]:
# Summarize key metrics from the archived results
def load_artifact(run_directory: Path, prefix: str, suffix: str) -> Path | None:
    pattern = f"{prefix}_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}_{suffix}"
    matches = list(run_directory.glob(pattern))
    return matches[0] if matches else None

aware_method_path = load_artifact(run_dir, 'label_aware', 'method_comparison.csv') if 'label_aware' in executed_prefixes else None
agnostic_method_path = load_artifact(run_dir, 'label_agnostic', 'method_comparison.csv') if 'label_agnostic' in executed_prefixes else None
aware_preds_path = load_artifact(run_dir, 'label_aware', 'pivot_predictions.csv') if 'label_aware' in executed_prefixes else None
agnostic_preds_path = load_artifact(run_dir, 'label_agnostic', 'pivot_predictions.csv') if 'label_agnostic' in executed_prefixes else None

summary_rows = []
for label, method_path, preds_path in [
    ('Label-Aware', aware_method_path, aware_preds_path),
    ('Label-Agnostic', agnostic_method_path, agnostic_preds_path)
]:
    if method_path is None or preds_path is None:
        print(f'Missing artifacts for {label}; skip summary.')
        continue
    method_df = pd.read_csv(method_path)
    preds_df = pd.read_csv(preds_path)
    if 'FastRP Embedding' not in method_df['Method'].values:
        print(f'FastRP results missing for {label}; skip summary.')
        continue
    fastrp_row = method_df[method_df['Method'] == 'FastRP Embedding'].iloc[0]
    pivot_rate = preds_df['became_pivot'].mean() * 100 if 'became_pivot' in preds_df.columns else np.nan
    summary_rows.append({
        'Mode': label,
        'Samples': len(preds_df),
        'Pivots': preds_df['became_pivot'].sum() if 'became_pivot' in preds_df.columns else np.nan,
        'Pivot Rate (%)': pivot_rate,
        'AUC-ROC': fastrp_row['AUC-ROC'],
        'AUC-PR': fastrp_row['AUC-PR'],
        'F1-Score': fastrp_row['F1-Score'],
        'Precision': fastrp_row['Precision'],
        'Recall': fastrp_row['Recall']
    })

def to_builtin(value):
    if isinstance(value, (np.generic,)):
        return value.item()
    return value

if summary_rows:
    summary_rows_builtin = [
        {key: to_builtin(value) for key, value in row.items()}
        for row in summary_rows
    ]
    summary_df = pd.DataFrame(summary_rows_builtin)
    print(summary_df.to_string(index=False))
    summary_payload = metadata.copy()
    summary_payload['metrics'] = summary_rows_builtin
    (run_dir / 'run_summary.json').write_text(json.dumps(summary_payload, indent=2))
else:
    print('No summary generated; verify artifacts above.')

In [ ]:
# Export CONNECTS edges for visualization artifacts
from CART.base import Neo4jConnection

CONNECTS_EXPORT_PATH = OUTPUT_DIR / 'connects_edges.csv'
export_cypher = """
CALL apoc.export.csv.query(
  "MATCH (a:IP)-[r:CONNECTS]->(b:IP)\n   RETURN a.address AS src,\n          b.address AS dst,\n          r.timestamp AS ts,\n          r.is_attack AS is_attack",
  'thesis_results/connects_edges.csv',
  {batchSize: 50000, delimiter: ',', quotes: false}
 )
YIELD file, source, format, nodes, relationships, properties, time
RETURN file, source, format, nodes, relationships, properties, time;
"""

needs_export = REFRESH_CONNECTS_EXPORT or not CONNECTS_EXPORT_PATH.exists()
if needs_export:
    print('Exporting IP→IP CONNECTS edges via APOC...')
    if not analyzer.connect():
        raise RuntimeError('SubnetPivotAnalyzer could not reconnect for export; check Neo4j status.')
    with analyzer.driver.session(database=analyzer.database) as session:
        check_query = (
            "SHOW PROCEDURES YIELD name "
            "WHERE name = 'apoc.export.csv.query' "
            "RETURN count(*) AS matches"
        )
        matches = session.run(check_query).single()["matches"]
        if matches == 0:
            raise RuntimeError("apoc.export.csv.query is not registered. Restart the Neo4j container with APOC enabled.")
        summary_records = session.run(export_cypher).data()
        if not summary_records:
            raise RuntimeError('APOC export returned no metadata; inspect Neo4j logs for details.')
        print('CSV export complete:')
        for key, value in summary_records[0].items():
            print(f'  {key}: {value}')
else:
    print(f'Using existing CONNECTS export at {CONNECTS_EXPORT_PATH}')

Neo4jConnection().ensure_export_permissions()
connects_df = pd.read_csv(CONNECTS_EXPORT_PATH, usecols=['src', 'dst', 'ts', 'is_attack'])
connects_df['ts'] = connects_df['ts'].astype('int64')
print(f'Loaded {len(connects_df):,} edges for downstream visualization.')

In [ ]:
# Build label-aware and label-agnostic visualizations with Polars pipelines
from typing import Literal, Optional

import polars as pl
try:
    import networkx as nx
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
except ImportError as exc:
    raise ImportError(
        "Install networkx and matplotlib before running visualization steps: `pip install networkx matplotlib`."
    ) from exc

ROLE_COLORS = {
    'attacker': '#e41a1c',
    'pivot': '#ffd60a',
    'victim_only': '#7b1fa2',
    'neutral': '#2ca02c',
}

ROLE_LABELS = {
    'attacker': 'Attacker',
    'pivot': 'Pivot (victim → attacker)',
    'victim_only': 'Victim only',
    'neutral': 'Non-victim/attacker',
}

CHAIN_SAMPLE_LIMIT = 200_000
MAX_EDGES_PER_NODE = 50
SUBNET_PREFIX = 24

def _prepare_ranked_edges(edges_lf: pl.LazyFrame, max_edges_per_node: Optional[int]) -> pl.LazyFrame:
    sorted_edges = edges_lf.sort('ts')
    ranked = sorted_edges.with_columns([
        pl.col('dst').cum_count().over('dst').alias('dst_rank'),
        pl.col('src').cum_count().over('src').alias('src_rank'),
    ])
    if max_edges_per_node is None:
        return ranked
    return ranked.filter(
        (pl.col('dst_rank') < max_edges_per_node) | (pl.col('src_rank') < max_edges_per_node)
    )

def _attach_subnets(df: pl.DataFrame, prefix: int) -> pl.DataFrame:
    if prefix != 24:
        raise ValueError('Only /24 subnets are supported right now.')
    pdf = df.to_pandas()
    for idx in range(1, 5):
        column = f'hop{idx}_ip'
        if column in pdf.columns:
            pdf[f'hop{idx}_subnet'] = pdf[column].str.rsplit('.', n=1).str[0] + '.0/24'
    return pl.from_pandas(pdf)

def build_multi_hop_chains(
    csv_path: str,
    *,
    mode: Literal['label_aware', 'label_agnostic'] = 'label_aware',
    max_edges_per_node: Optional[int] = MAX_EDGES_PER_NODE,
    max_results: Optional[int] = CHAIN_SAMPLE_LIMIT,
    streaming: bool = True,
    drop_duplicate_chains: bool = True,
    attach_subnets: bool = True,
    save_csv: Optional[str] = None,
    subnet_prefix: int = SUBNET_PREFIX,
    return_total: bool = False,
 ) -> pl.DataFrame:
    source_path = Path(csv_path)
    if not source_path.exists():
        raise FileNotFoundError(f'CSV file not found: {source_path}')
    if not os.access(source_path, os.R_OK):
        Neo4jConnection().ensure_export_permissions()
        if not os.access(source_path, os.R_OK):
            raise PermissionError(f'Unable to read {source_path}; adjust permissions and retry.')

    lf = pl.scan_csv(str(source_path), infer_schema_length=1000).select(['src', 'dst', 'ts', 'is_attack'])
    if mode == 'label_aware':
        lf = lf.filter(pl.col('is_attack') == 1)

    lf = lf.with_columns(pl.col('ts').cast(pl.Int64))
    ranked = _prepare_ranked_edges(lf, max_edges_per_node)

    hop1 = ranked.select([
        pl.col('src').alias('hop1_ip'),
        pl.col('dst').alias('hop2_ip'),
        pl.col('ts').alias('t1'),
    ])
    hop2 = ranked.select([
        pl.col('src').alias('hop2_ip'),
        pl.col('dst').alias('hop3_ip'),
        pl.col('ts').alias('t2'),
    ])
    hop3 = ranked.select([
        pl.col('src').alias('hop3_ip'),
        pl.col('dst').alias('hop4_ip'),
        pl.col('ts').alias('t3'),
    ])

    chains = (
        hop1.join(hop2, on='hop2_ip', how='inner')
            .filter(pl.col('t2') > pl.col('t1'))
            .join(hop3, on='hop3_ip', how='inner')
            .filter(pl.col('t3') > pl.col('t2'))
            .filter(pl.col('hop1_ip') != pl.col('hop3_ip'))
            .filter(pl.col('hop2_ip') != pl.col('hop4_ip'))
            .filter(pl.col('hop1_ip') != pl.col('hop4_ip'))
            .with_columns([
                ((pl.col('t2') - pl.col('t1')) / 3600.0).alias('hours_to_hop2'),
                ((pl.col('t3') - pl.col('t2')) / 3600.0).alias('hours_to_hop3'),
            ])
    )

    engine = 'streaming' if streaming else None
    try:
        total_count = chains.select(pl.len().alias('count')).collect(engine=engine)[0, 'count']
    except pl.exceptions.ComputeError:
        total_count = chains.select(pl.len().alias('count')).collect(engine=None)[0, 'count']

    working_chains = chains.limit(max_results) if max_results is not None else chains

    try:
        result = working_chains.collect(engine=engine)
    except pl.exceptions.ComputeError:
        result = working_chains.collect(engine=None)

    if attach_subnets:
        result = _attach_subnets(result, subnet_prefix)

    if drop_duplicate_chains:
        result = result.unique()

    if save_csv:
        result.write_csv(save_csv)

    if return_total:
        return result, int(total_count)
    return result

def summarize_chains(df: pl.DataFrame, label: str, total: Optional[int] = None) -> None:
    if df.is_empty():
        print(f'{label}: no chains found')
        return
    pdf = df.to_pandas()
    total_count = int(total) if total is not None else df.height
    unique_ips = {
        f'hop{idx}_ip': pdf[f'hop{idx}_ip'].nunique() for idx in range(1, 5) if f'hop{idx}_ip' in pdf
    }
    unique_subnets = {
        f'hop{idx}_subnet': pdf[f'hop{idx}_subnet'].nunique() for idx in range(1, 5) if f'hop{idx}_subnet' in pdf
    }
    print(f'{label}: {total_count:,} chains')
    if unique_ips:
        print('  Unique IPs per hop:')
        for key, value in unique_ips.items():
            print(f'    {key}: {value:,}')
    if unique_subnets:
        print('  Unique subnets per hop:')
        for key, value in unique_subnets.items():
            print(f'    {key}: {value:,}')
    print('  Sample chains:')
    display(pdf.head(5))

def _ip_to_subnet(ip: str, prefix: int = SUBNET_PREFIX) -> str:
    if prefix != 24:
        raise ValueError('Only /24 subnets are supported right now.')
    parts = ip.split('.')
    if len(parts) != 4:
        return ip
    return '.'.join(parts[:3]) + '.0/24'

def _classify_roles(
    df: pd.DataFrame,
    *,
    aggregate_by_subnet: bool = False,
    subnet_prefix: int = SUBNET_PREFIX,
 ) -> tuple[set[str], set[str], set[str], set[str]]:
    attack_edges = df[df['is_attack'] == 1]
    attack_sources = set(attack_edges['src'])
    attack_targets = set(attack_edges['dst'])
    pivot_nodes = attack_sources & attack_targets
    attacker_only = attack_sources - pivot_nodes
    victim_only = attack_targets - attack_sources
    all_nodes = set(df['src']).union(set(df['dst']))
    neutral_nodes = all_nodes - attack_sources - attack_targets

    if not aggregate_by_subnet:
        return attacker_only, pivot_nodes, victim_only, neutral_nodes

    def _transform(nodes: set[str]) -> set[str]:
        return {_ip_to_subnet(node, subnet_prefix) for node in nodes}

    return (
        _transform(attacker_only),
        _transform(pivot_nodes),
        _transform(victim_only),
        _transform(neutral_nodes),
    )

def _subset_for_visualization(
    df: pd.DataFrame,
    *,
    include_neutral: bool,
    max_nodes: Optional[int],
    max_edges: Optional[int],
    aggregate_by_subnet: bool,
    subnet_prefix: int,
 ) -> tuple[pd.DataFrame, set[str], dict[str, str]]:
    attacker_only, pivot_nodes, victim_only, neutral_nodes = _classify_roles(
        df, aggregate_by_subnet=aggregate_by_subnet, subnet_prefix=subnet_prefix
    )
    attack_related_nodes = attacker_only | pivot_nodes | victim_only

    working_df = df.copy()
    if not include_neutral:
        working_df = working_df[
            working_df['src'].isin(attack_related_nodes) & working_df['dst'].isin(attack_related_nodes)
        ]

    if working_df.empty:
        return working_df, set(), {}

    working_df = working_df.sort_values(['is_attack', 'ts'], ascending=[False, True])

    if aggregate_by_subnet:
        working_df = working_df.assign(
            src=working_df['src'].map(lambda ip: _ip_to_subnet(ip, subnet_prefix)),
            dst=working_df['dst'].map(lambda ip: _ip_to_subnet(ip, subnet_prefix)),
        )
        grouped = working_df.groupby(['src', 'dst'], as_index=False).agg(
            ts=('ts', 'min'),
            is_attack=('is_attack', 'max'),
            edge_count=('ts', 'count'),
            attack_edges=('is_attack', 'sum'),
        )
        grouped['is_attack'] = grouped['is_attack'].astype(int)
        working_df = grouped.sort_values(['is_attack', 'ts'], ascending=[False, True])

    if max_edges is not None and len(working_df) > max_edges:
        print(
            f'Truncating edges to top {max_edges} of {len(working_df)} rows (prioritised by attack flag, then time).'
        )
        working_df = working_df.head(max_edges)

    degree_counts = (
        pd.concat([working_df['src'], working_df['dst']]).value_counts().rename('degree')
    )
    node_subset: set[str] = set(degree_counts.index)

    if max_nodes is not None and len(node_subset) > max_nodes:
        def _priority(node: str) -> tuple[int, int]:
            if node in pivot_nodes:
                role_rank = 0
            elif node in attacker_only:
                role_rank = 1
            elif node in victim_only:
                role_rank = 2
            else:
                role_rank = 3
            return role_rank, -int(degree_counts.get(node, 0))

        ordered_nodes = sorted(node_subset, key=_priority)
        kept_nodes = ordered_nodes[:max_nodes]
        print(
            f'Truncating nodes to {max_nodes} of {len(node_subset)} (prioritising pivots, attackers, victims, then neutrals by degree).'
        )
        node_subset = set(kept_nodes)
        working_df = working_df[
            working_df['src'].isin(node_subset) & working_df['dst'].isin(node_subset)
        ]

    if working_df.empty:
        return working_df, set(), {}

    node_subset = set(working_df['src']).union(set(working_df['dst']))

    role_map: dict[str, str] = {}
    for node in node_subset:
        if node in pivot_nodes:
            role_map[node] = 'pivot'
        elif node in attacker_only:
            role_map[node] = 'attacker'
        elif node in victim_only:
            role_map[node] = 'victim_only'
        else:
            role_map[node] = 'neutral'

    return working_df, node_subset, role_map

def build_attack_graph(
    df: pd.DataFrame,
    *,
    include_neutral: bool = True,
    max_nodes: Optional[int] = None,
    max_edges: Optional[int] = None,
    aggregate_by_subnet: bool = False,
    subnet_prefix: int = SUBNET_PREFIX,
 ) -> tuple[nx.DiGraph, dict[str, str], pd.DataFrame]:
    edges_df, node_subset, role_map = _subset_for_visualization(
        df,
        include_neutral=include_neutral,
        max_nodes=max_nodes,
        max_edges=max_edges,
        aggregate_by_subnet=aggregate_by_subnet,
        subnet_prefix=subnet_prefix,
    )

    graph = nx.DiGraph()
    for node, role in role_map.items():
        graph.add_node(node, role=role)
    for row in edges_df.itertuples(index=False):
        edge_attrs = {'is_attack': int(getattr(row, 'is_attack', 0))}
        if hasattr(row, 'edge_count'):
            edge_attrs['edge_count'] = getattr(row, 'edge_count')
        if hasattr(row, 'attack_edges'):
            edge_attrs['attack_edges'] = getattr(row, 'attack_edges')
        graph.add_edge(row.src, row.dst, **edge_attrs)

    return graph, role_map, edges_df

def draw_attack_graph(
    graph: nx.DiGraph,
    role_map: dict[str, str],
    *,
    title: str,
    layout: str = 'spring',
    seed: int = 42,
    figsize: tuple[int, int] = (16, 12),
    node_size: int = 60,
    edge_alpha: float = 0.18,
    edge_weight_attr: Optional[str] = None,
    attack_edge_width: float = 1.8,
    benign_edge_width: float = 0.6,
    save_path: Optional[Path] = None,
 ) -> None:
    if graph.number_of_nodes() == 0:
        print('No nodes available to draw; adjust filtering parameters.')
        return

    if layout == 'spring':
        pos = nx.spring_layout(graph, seed=seed)
    elif layout == 'kamada_kawai':
        pos = nx.kamada_kawai_layout(graph)
    else:
        pos = nx.random_layout(graph, seed=seed)

    plt.figure(figsize=figsize)
    for role, color in ROLE_COLORS.items():
        nodes = [n for n in graph.nodes if role_map.get(n) == role]
        if not nodes:
            continue
        nx.draw_networkx_nodes(
            graph,
            pos,
            nodelist=nodes,
            node_color=color,
            node_size=node_size,
            alpha=0.85,
            label=ROLE_LABELS[role],
        )

    attack_edges = [(u, v) for u, v, d in graph.edges(data=True) if d.get('is_attack') == 1]
    benign_edges = [(u, v) for u, v, d in graph.edges(data=True) if d.get('is_attack') != 1]

    def _edge_widths(edge_list: list[tuple[str, str]], base_width: float) -> list[float]:
        if not edge_list:
            return []
        if edge_weight_attr is None:
            return [base_width] * len(edge_list)
        weights = [float(graph[u][v].get(edge_weight_attr, 1.0)) for u, v in edge_list]
        max_weight = max(weights)
        if max_weight <= 0:
            return [base_width] * len(edge_list)
        min_width = max(base_width * 0.35, 0.35)
        scale = base_width / max_weight
        return [max(weight * scale, min_width) for weight in weights]

    if benign_edges:
        nx.draw_networkx_edges(
            graph,
            pos,
            edgelist=benign_edges,
            edge_color='#94a3b8',
            alpha=edge_alpha,
            arrows=False,
            width=_edge_widths(benign_edges, benign_edge_width),
        )
    if attack_edges:
        nx.draw_networkx_edges(
            graph,
            pos,
            edgelist=attack_edges,
            edge_color='#ff595e',
            alpha=max(edge_alpha, 0.3),
            arrows=False,
            width=_edge_widths(attack_edges, attack_edge_width),
        )

    plt.title(title)
    plt.axis('off')
    legend_handles = [
        Line2D([0], [0], marker='o', color='w', label=ROLE_LABELS[role],
               markerfacecolor=color, markersize=10)
        for role, color in ROLE_COLORS.items()
    ]
    plt.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=False)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

def _add_chain_to_graph(graph: nx.DiGraph, chain: dict) -> None:
    nodes = [chain.get(f'hop{idx}_ip') for idx in range(1, 5)]
    nodes = [node for node in nodes if node]
    for hop_index, node in enumerate(nodes, start=1):
        if node not in graph:
            graph.add_node(node, hop=hop_index)
    for idx in range(len(nodes) - 1):
        graph.add_edge(nodes[idx], nodes[idx + 1])

HOP_COLOR_MAP = {1: '#1f77b4', 2: '#ff7f0e', 3: '#2ca02c', 4: '#d62728'}

def visualize_chain_network(
    df: pl.DataFrame,
    label: str,
    limit: int = 250,
    *,
    layout_seed: int = 42,
    figsize: tuple[int, int] = (14, 10),
    show_node_labels: bool = False,
    save_path: Optional[Path] = None,
 ) -> None:
    if df.is_empty():
        print(f'{label}: no chains available for visualization')
        return

    sample_pdf = df.head(limit).to_pandas()
    graph = nx.DiGraph()
    for _, row in sample_pdf.iterrows():
        _add_chain_to_graph(graph, row.to_dict())

    if graph.number_of_edges() == 0:
        print(f'{label}: sample contained no edges after filtering')
        return

    hop_colors = [
        HOP_COLOR_MAP.get(graph.nodes[node].get('hop', 0), '#7f7f7f') for node in graph.nodes
    ]
    pos = nx.spring_layout(graph, seed=layout_seed, k=0.6)
    plt.figure(figsize=figsize)
    nx.draw_networkx_nodes(graph, pos, node_color=hop_colors, node_size=120, alpha=0.85)
    nx.draw_networkx_edges(graph, pos, arrowstyle='-|>', arrowsize=12, alpha=0.6)
    if show_node_labels:
        nx.draw_networkx_labels(graph, pos, font_size=8, font_color='black')
    plt.title(f"{label} chains (sampled {min(limit, len(sample_pdf))} of {df.height:,})")
    legend_handles = [
        plt.Line2D([0], [0], marker='o', color='w', label=f'Hop {idx}',
                   markerfacecolor=color, markersize=10)
        for idx, color in HOP_COLOR_MAP.items()
    ]
    plt.legend(handles=legend_handles, loc='upper right')
    plt.axis('off')

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

import os

chain_prefix = f"h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}"
label_aware_chain_path = run_dir / f'label_aware_{chain_prefix}_chains.csv'
label_agnostic_chain_path = run_dir / f'label_agnostic_{chain_prefix}_chains.csv'

label_aware_chains, label_aware_total = build_multi_hop_chains(
    str(CONNECTS_EXPORT_PATH),
    mode='label_aware',
    max_edges_per_node=MAX_EDGES_PER_NODE,
    max_results=CHAIN_SAMPLE_LIMIT,
    streaming=True,
    save_csv=str(label_aware_chain_path),
    return_total=True,
 )
summarize_chains(label_aware_chains, 'Label-aware', total=label_aware_total)

label_agnostic_chains, label_agnostic_total = build_multi_hop_chains(
    str(CONNECTS_EXPORT_PATH),
    mode='label_agnostic',
    max_edges_per_node=MAX_EDGES_PER_NODE,
    max_results=CHAIN_SAMPLE_LIMIT,
    streaming=True,
    save_csv=str(label_agnostic_chain_path),
    return_total=True,
 )
summarize_chains(label_agnostic_chains, 'Label-agnostic', total=label_agnostic_total)

visualize_chain_network(
    label_aware_chains,
    'Label-aware',
    limit=200,
    show_node_labels=False,
    save_path=run_dir / f'label_aware_{chain_prefix}_chain_network.png',
 )
visualize_chain_network(
    label_agnostic_chains,
    'Label-agnostic',
    limit=200,
    show_node_labels=False,
    save_path=run_dir / f'label_agnostic_{chain_prefix}_chain_network.png',
 )

label_aware_df = connects_df[connects_df['is_attack'] == 1].copy()
if label_aware_df.empty:
    print('Label-aware dataset produced no attack edges; skipping network visualizations.')
else:
    la_graph, la_role_map, la_edges = build_attack_graph(
        label_aware_df,
        include_neutral=True,
        max_nodes=None,
        max_edges=None,
    )
    print(
        f"Label-aware IP graph: {la_graph.number_of_nodes():,} nodes, {la_graph.number_of_edges():,} edges "
        f"(attack edges = {(la_edges['is_attack'] == 1).sum():,})."
    )
    draw_attack_graph(
        la_graph,
        la_role_map,
        title='Label-aware attack flow (IP-level)',
        layout='spring',
        seed=24,
        figsize=(20, 16),
        node_size=35,
        edge_alpha=0.12,
        save_path=run_dir / f'label_aware_{chain_prefix}_ip_graph.png',
    )

    la_subnet_graph, la_subnet_role_map, la_subnet_edges = build_attack_graph(
        label_aware_df,
        include_neutral=True,
        max_nodes=None,
        max_edges=None,
        aggregate_by_subnet=True,
        subnet_prefix=SUBNET_PREFIX,
    )
    total_la_edges = (
        int(la_subnet_edges['edge_count'].sum())
        if 'edge_count' in la_subnet_edges.columns
        else len(la_subnet_edges)
    )
    print(
        f"Label-aware subnet graph: {la_subnet_graph.number_of_nodes():,} subnets, {la_subnet_graph.number_of_edges():,} edges "
        f"(underlying edges represented = {total_la_edges:,})."
    )
    draw_attack_graph(
        la_subnet_graph,
        la_subnet_role_map,
        title='Label-aware attack flow (subnet /24)',
        layout='spring',
        seed=32,
        figsize=(18, 14),
        node_size=140,
        edge_alpha=0.25,
        edge_weight_attr='edge_count',
        attack_edge_width=2.6,
        benign_edge_width=1.2,
        save_path=run_dir / f'label_aware_{chain_prefix}_subnet_graph.png',
    )

full_graph, full_role_map, full_edges = build_attack_graph(
    connects_df,
    include_neutral=True,
    max_nodes=None,
    max_edges=None,
 )
print(
    f"Label-agnostic IP graph: {full_graph.number_of_nodes():,} nodes, {full_graph.number_of_edges():,} edges "
    f"(attack edges = {(full_edges['is_attack'] == 1).sum():,})."
 )
draw_attack_graph(
    full_graph,
    full_role_map,
    title='Label-agnostic attack flow (IP-level)',
    layout='spring',
    seed=20,
    figsize=(20, 16),
    node_size=32,
    edge_alpha=0.1,
    save_path=run_dir / f'label_agnostic_{chain_prefix}_ip_graph.png',
 )

subnet_graph, subnet_role_map, subnet_edges = build_attack_graph(
    connects_df,
    include_neutral=True,
    max_nodes=None,
    max_edges=None,
    aggregate_by_subnet=True,
    subnet_prefix=SUBNET_PREFIX,
 )
total_edges = (
    int(subnet_edges['edge_count'].sum()) if 'edge_count' in subnet_edges.columns else len(subnet_edges)
 )
print(
    f"Label-agnostic subnet graph: {subnet_graph.number_of_nodes():,} subnets, {subnet_graph.number_of_edges():,} edges "
    f"(underlying edges represented = {total_edges:,})."
 )
draw_attack_graph(
    subnet_graph,
    subnet_role_map,
    title='Label-agnostic attack flow (subnet /24)',
    layout='spring',
    seed=28,
    figsize=(18, 14),
    node_size=135,
    edge_alpha=0.25,
    edge_weight_attr='edge_count',
    attack_edge_width=2.6,
    benign_edge_width=1.2,
    save_path=run_dir / f'label_agnostic_{chain_prefix}_subnet_graph.png',
 )

print('\nSaved visualization artifacts:')
for artifact in sorted(run_dir.glob('*.png')):
    print(f'  {artifact.name}')
for artifact in sorted(run_dir.glob('*.csv')):
    if artifact.name.endswith('_chains.csv'):
        print(f'  {artifact.name}')

In [ ]:
# Optional: stop the container when finished
controller.close()
if SHUTDOWN_AFTER_RUN:
    controller.stop()
    print('Neo4j container stopped.')
else:
    print('Neo4j container left running; call controller.stop() if you want to shut it down.')

In [ ]:
# Generate all thesis artifacts (tables, figures, summaries)
if GENERATE_THESIS_ARTIFACTS:
    print('\n' + '=' * 80)
    print('GENERATING THESIS ARTIFACTS')
    print('=' * 80)
    import subprocess
    import sys
    
    result = subprocess.run(
        [sys.executable, 'generate_all_thesis_artifacts.py'],
        cwd=str(Path.cwd()),
        capture_output=True,
        text=True
    )
    
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    if result.returncode == 0:
        print('\n✓ Thesis artifacts generated successfully!')
        print('  Check thesis_figures/ for all outputs')
    else:
        print(f'\n⚠ Artifact generation completed with code {result.returncode}')
else:
    print('\nSkipping artifact generation (set GENERATE_THESIS_ARTIFACTS=True to enable)')